<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.2-calling-loop/practice/GCP_Capstone_6.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 6.2 — Complete Calling Loop

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the unified `google-genai` SDK, authenticate with Application Default Credentials (Colab), and initialize a Vertex AI client. Run this cell first — every exercise below depends on `client`, the DocuMind functions, and `TOOLS`.

In [ ]:
%%bash
pip install -q google-genai

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE to your project id

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='us-central1')
print(f'Client ready for {PROJECT_ID}')

### DocuMind functions + tool list

Three mock DocuMind tools the model can call. `TOOLS` is the callable list passed to `GenerateContentConfig(tools=...)`; `FUNCTIONS` is the name→callable map the dispatcher uses to execute a chosen call.

In [ ]:
def search_documents(query: str, doc_type: str = 'all', top_k: int = 5) -> dict:
    """Search DocuMind document collection by query.
    Args:
        query: Search query in natural language
        doc_type: Filter by type (research_paper, invoice, legal, form, all)
        top_k: Number of results to return
    """
    mock = {
        'legal': [{'id': 'D-01', 'title': 'NDA Template', 'pages': 8},
                  {'id': 'D-02', 'title': 'Service Agreement', 'pages': 24},
                  {'id': 'D-03', 'title': 'Employment Contract', 'pages': 15}],
        'invoice': [{'id': 'D-10', 'title': 'Q1 Invoice', 'pages': 2}],
    }
    if doc_type == 'all':
        results = [d for docs in mock.values() for d in docs]
    else:
        results = mock.get(doc_type, [])
    return {'documents': results[:top_k], 'total': len(results)}

def calculate_processing_cost(num_documents: int, total_pages: int,
                              processing_type: str = 'standard') -> dict:
    """Estimate document processing cost in USD and INR.
    Args:
        num_documents: Number of documents to process
        total_pages: Total page count across all documents
        processing_type: Tier — standard, priority, or bulk
    """
    rates = {'standard': 0.05, 'priority': 0.12, 'bulk': 0.03}
    cost = total_pages * rates.get(processing_type, 0.05)
    return {'num_documents': num_documents, 'total_pages': total_pages,
            'rate_per_page': rates.get(processing_type, 0.05),
            'cost_usd': round(cost, 2), 'cost_inr': round(cost * 85, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind RAG pipeline usage statistics.
    Args:
        metric: Which metric (queries, costs, latency, users)
        days: Number of days to look back
    """
    data = {'queries': 1247, 'costs': 18.50, 'latency': 245, 'users': 42}
    return {'metric': metric, 'period': f'last {days} days',
            'value': data.get(metric, 0), 'trend': '+12%'}

FUNCTIONS = {
    'search_documents': search_documents,
    'calculate_processing_cost': calculate_processing_cost,
    'get_usage_stats': get_usage_stats,
}
TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]
print(f'Registered {len(FUNCTIONS)} functions')

## Exercise 1: Manual 2-Turn Loop

**Difficulty:** Easy

Build the contents list manually. Send, append model content, execute, append result, send again.

1. Create initial contents with user prompt
2. Send, append `response.candidates[0].content`
3. Execute function, append `FunctionResponse` with id
4. Send again, get final text

In [ ]:
# Do the loop by hand for ONE call so you can see each step of the round-trip.
config = types.GenerateContentConfig(tools=TOOLS)

# Step 1: initial contents with the user prompt
contents = [types.Content(role='user', parts=[
    types.Part.from_text(text='How many queries did we get this week?')])]

# --- Turn 1: model decides to call a function ---
resp1 = client.models.generate_content(
    model='gemini-3.6-flash', contents=contents, config=config)

# Step 2: append the model's content (carries the function_call + thought signature)
contents.append(resp1.candidates[0].content)

fc = resp1.function_calls[0]
print(f'Model called: {fc.name}({dict(fc.args)})')

# Step 3: execute the function, append the result as a FunctionResponse part
result = FUNCTIONS[fc.name](**fc.args)
contents.append(types.Content(role='user', parts=[
    types.Part.from_function_response(
        name=fc.name, response={'result': result})]))

# --- Turn 2: send the result back, model synthesizes final text ---
resp2 = client.models.generate_content(
    model='gemini-3.6-flash', contents=contents, config=config)
print('\n=== Final text ===')
print(resp2.text)

## Exercise 2: While-Loop Dispatcher

**Difficulty:** Easy

Implement `run_function_loop`. Test with a single-call query like "How many queries this week?"

1. While loop with `max_turns`
2. Check `response.function_calls`
3. Execute all, append results
4. Verify it exits after a text response

In [ ]:
def run_function_loop(client, prompt, tools, functions,
                      model='gemini-3.6-flash',
                      system_instruction='', max_turns=10):
    """Production while-loop: call until the model returns text."""
    config = types.GenerateContentConfig(
        tools=tools, system_instruction=system_instruction)
    contents = [types.Content(role='user', parts=[
        types.Part.from_text(text=prompt)])]

    for turn in range(max_turns):
        response = client.models.generate_content(
            model=model, contents=contents, config=config)
        contents.append(response.candidates[0].content)

        if not response.function_calls:
            return response.text  # Done!

        result_parts = []
        for fc in response.function_calls:
            print(f'  [Turn {turn+1}] {fc.name}({dict(fc.args)})')
            try:
                result = functions[fc.name](**fc.args)
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response={'result': result}))
            except Exception as e:
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response={'error': str(e)}))

        contents.append(types.Content(role='user', parts=result_parts))

    return 'Max turns reached.'

print('While-loop dispatcher ready')

In [ ]:
# Single-call test: one function call, then one text response (loop exits).
answer = run_function_loop(
    client=client,
    prompt='How many queries did we get this week?',
    tools=TOOLS,
    functions=FUNCTIONS)
print('\n=== Answer ===')
print(answer)

## Exercise 3: Error-as-Data

**Difficulty:** Easy

Make a function return an error dict. Verify the model explains the error naturally.

1. Return `{"error": "Database unavailable"}`
2. Send via `FunctionResponse`
3. Verify the model explains gracefully

In [ ]:
# Error-as-data: the tool doesn't raise — it RETURNS an error payload.
# The model reads it and explains the failure in plain language.
def search_documents_down(query: str, doc_type: str = 'all', top_k: int = 5) -> dict:
    """Search DocuMind documents (currently returns a backend error).
    Args:
        query: Search query in natural language
        doc_type: Filter by type (research_paper, invoice, legal, form, all)
        top_k: Number of results to return
    """
    return {'error': 'Database unavailable'}

answer = run_function_loop(
    client=client,
    prompt='Find our legal documents',
    tools=[search_documents_down],
    functions={'search_documents_down': search_documents_down},
    system_instruction='You are DocuMind AI. If a tool returns an error, '
                       'explain the situation to the user calmly and suggest '
                       'they retry shortly. Never invent document data.')
print('\n=== Answer ===')
print(answer)

## Exercise 4: Sequential Chain

**Difficulty:** Medium

"Find legal docs and calculate cost." Verify a 2-turn chain: search → calculate.

1. Prompt requires search results before cost calc
2. Trace: Turn 1 = search, Turn 2 = calculate
3. Final text synthesizes both results

In [ ]:
# Sequential: the model must search FIRST to get real page counts,
# then feed those into the cost calculation. Watch the [Turn N] trace.
answer = run_function_loop(
    client=client,
    prompt='Find all legal documents and estimate bulk processing cost',
    tools=TOOLS,
    functions=FUNCTIONS,
    system_instruction='You are DocuMind AI. When estimating costs, first '
                       'search for documents to get accurate page counts. '
                       'Never guess page numbers. If search returns no '
                       'results, explain this clearly.')
print('\n=== Sequential Chain Answer ===')
print(answer)

## Exercise 5: Parallel Execution

**Difficulty:** Medium

Trigger 2 independent calls. Execute concurrently. Send both results. Verify synthesis.

1. "Search invoices AND show query stats"
2. Check `len(response.function_calls) == 2`
3. Execute both, send both `FunctionResponse` objects

In [ ]:
# Two INDEPENDENT asks in one prompt -> the model emits two function_calls in a
# SINGLE response. run_function_loop already executes every call in the turn and
# returns all FunctionResponse parts together. First, confirm the parallel emit:
config = types.GenerateContentConfig(
    tools=TOOLS,
    system_instruction='You are DocuMind AI. Use tools to answer accurately.')
probe = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Search our invoices AND show me this week\'s query stats',
    config=config)
print('Parallel calls in one response:', len(probe.function_calls))
for fc in probe.function_calls:
    print(f'  {fc.name}({dict(fc.args)})')

# Now let the dispatcher run both and synthesize a single answer.
answer = run_function_loop(
    client=client,
    prompt='Search our invoices AND show me this week\'s query stats',
    tools=TOOLS,
    functions=FUNCTIONS,
    system_instruction='You are DocuMind AI. Use tools to answer accurately.')
print('\n=== Parallel Answer ===')
print(answer)

## Exercise 6: FunctionRegistry

**Difficulty:** Medium

Build a registry with 3 functions, timeouts, and destructive-op blocking. Test all paths.

1. `register()` 3 functions with different timeouts
2. `execute()` a valid function → result
3. `execute()` a blocked function → error
4. `execute()` an unknown function → error

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
import logging

class FunctionRegistry:
    BLOCKED = {'delete_document', 'send_email', 'modify_access'}

    def __init__(self):
        self._funcs = {}
        self._timeouts = {}

    def register(self, name, func, timeout=30):
        self._funcs[name] = func
        self._timeouts[name] = timeout

    def execute(self, name, args):
        if name in self.BLOCKED:
            return {'error': f'{name} requires manual approval'}
        if name not in self._funcs:
            return {'error': f'Unknown function: {name}'}
        try:
            with ThreadPoolExecutor(max_workers=1) as pool:
                future = pool.submit(self._funcs[name], **args)
                result = future.result(timeout=self._timeouts[name])
            return {'result': result}
        except FuturesTimeout:
            return {'error': f'{name} timed out after {self._timeouts[name]}s'}
        except Exception as e:
            return {'error': f'Failed: {str(e)}'}

# Test all four paths
reg = FunctionRegistry()
reg.register('search_documents', search_documents, timeout=30)
reg.register('calculate_processing_cost', calculate_processing_cost, timeout=10)
reg.register('get_usage_stats', get_usage_stats, timeout=60)

print('Registry tests:')
print(f"  success: {reg.execute('search_documents', {'query': 'test'})}")
print(f"  blocked: {reg.execute('delete_document', {'id': 'D-01'})}")
print(f"  unknown: {reg.execute('nonexistent', {})}")
print(f"  bad args: {reg.execute('calculate_processing_cost', {'invalid': True})}")

## Exercise 7: Chat + Auto Calling

**Difficulty:** Challenge

Build a chat session. Test a multi-turn conversation with sequential tool use and context.

1. `client.chats.create` with tools
2. Turn 1: search. Turn 2: cost based on search. Turn 3: stats.
3. Turn 4-5: follow-ups using earlier context

In [ ]:
# chats.create runs the function-calling loop AUTOMATICALLY per send_message.
# History (including earlier tool results) carries across turns for free.
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer '
                           'document questions accurately.'))

def show(label, resp):
    t = resp.text
    print(f'{label}: {t[:150]}...' if len(t) > 150 else f'{label}: {t}')

show('Turn 1', chat.send_message('What legal documents do we have?'))
show('Turn 2', chat.send_message('How much would priority processing cost for those?'))
show('Turn 3', chat.send_message("Show me this month's query stats"))
# Follow-ups that lean on earlier context
show('Turn 4', chat.send_message('Which of those legal docs has the most pages?'))
show('Turn 5', chat.send_message('Compare that priority cost to bulk pricing.'))

print(f'\nTotal history messages: {len(chat.get_history())}')

## Exercise 8: DocuMindAgent Class

**Difficulty:** Challenge

Build a `DocuMindAgent` with `ask()` for single-turn and `create_chat()` for multi-turn.

1. Class with registry, system prompt, tools
2. `ask()` uses `run_function_loop`
3. `create_chat()` returns an auto-calling session
4. Test both modes

In [ ]:
class DocuMindAgent:
    SYSTEM_PROMPT = (
        'You are DocuMind AI, a document intelligence assistant. '
        'Today is 2026-04-15. Use tools for document questions. '
        'When estimating costs, first search for real document counts. '
        'If search returns no results, explain clearly.')

    def __init__(self, client):
        self.client = client
        self.tools = [search_documents, calculate_processing_cost,
                      get_usage_stats]
        self.functions = FUNCTIONS

    def ask(self, question):
        return run_function_loop(
            client=self.client, prompt=question,
            tools=self.tools, functions=self.functions,
            system_instruction=self.SYSTEM_PROMPT, max_turns=8)

    def create_chat(self):
        return self.client.chats.create(
            model='gemini-3.6-flash',
            config=types.GenerateContentConfig(
                tools=self.tools,
                system_instruction=self.SYSTEM_PROMPT))

# Test single-turn (sequential chain via run_function_loop)
agent = DocuMindAgent(client)
print('=== Single-turn ===')
print(agent.ask('Find all legal documents and estimate standard processing cost'))

# Test multi-turn (auto-calling chat)
print('\n=== Multi-turn ===')
chat = agent.create_chat()
print(chat.send_message('What invoices do we have?').text)
print(chat.send_message('How much to process them in bulk?').text)